# M5 Forecasting — Feature Engineering

**Project:** Retail-Demand-Forecasting
**Stage:** `05_feature_engineering` — leakage-safe, model-ready feature dataset
**Status:** Research/experimentation notebook, but its output feeds real training
**Author:** Data Engineering / Analytics Team

---

## Where this notebook sits in the project

```
PostgreSQL (normalized: calendar, products, stores, prices, sales)
    │
    ▼
data/processed/training_dataset/train_dataset.parquet   ← built by the previous notebook,
    │                                                       via a database-side join + chunked
    │                                                       retrieval, for a controlled subset
    │                                                       (5 stores, 300 products, 2013-01-01
    │                                                       to 2015-01-01, ~1M rows)
    ▼
[ THIS NOTEBOOK ]
    │
    ├─ Calendar features
    ├─ Price features
    ├─ Lag features            (strictly historical)
    ├─ Rolling features        (strictly historical, shifted before rolling)
    ├─ Leakage checks
    ├─ Chronological train / validation / test split
    ▼
data/processed/features/{train,validation,test}.parquet   ← model-ready output
    │
    ▼
Model training (XGBoost or similar — NOT part of this notebook)
```

## Objective

Turn the flat analytical subset (`train_dataset.parquet`) into a **model-ready feature
dataset** — calendar, price, lag, and rolling features — split chronologically into
train/validation/test. This is a working MVP, not a Kaggle-grade feature set: a small number
of well-understood, well-validated features, built correctly, beats a large number of
features built carelessly.

### The one rule that overrides every other decision in this notebook

> **No feature at time `t` may use information that would not actually be available at time
> `t` in a real forecasting scenario.** Concretely: no feature is allowed to look at
> `sales_quantity` (or anything derived from it) at time `t` or later — only at `t-1` and
> earlier. Every lag/rolling feature below is built and then explicitly checked against this
> rule (Section 10).

### Explicit scope boundaries

- **No model training** — this notebook's output is training input for a future notebook, not
  a trained model.
- **No elaborate feature-engineering framework** — no classes, no configuration system, just
  clear, sequential, inspectable `pandas` operations.
- **No row sampling** — the continuous time-series structure is preserved throughout; rows are
  never shuffled or randomly subset.
- **No Spark/Dask/Polars/Airflow** — plain `pandas` + `pyarrow`/Parquet, matching the rest of
  this project.


## 2. Imports and Configuration

**What we're doing:** Importing `pandas`/`numpy` for the feature work, `pathlib` for portable
paths, and `json` for the small metadata file saved at the end. We also resolve the project
root the same way every notebook in this project does — a marker-based upward search — so this
notebook runs correctly regardless of the current working directory.

**Why we're doing it this way:** Consistency with the Extract/Transform/Load/dataset-builder
notebooks matters here specifically because this notebook is meant to be run *after* them, in
the same project — reusing the same path-resolution pattern means no additional setup is
needed.

**What we expect:** A resolved project root, and two paths: the source Parquet file this
notebook reads, and the output directory it will eventually write to.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# Marker-based project root resolution -- same pattern used throughout this project.
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "data" / "processed").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate a 'data/processed' directory above {current}. "
        "Run this notebook from inside the Retail-Demand-Forecasting project."
    )

SOURCE_PATH = project_root / "data" / "processed" / "training_dataset" / "train_dataset.parquet"
FEATURES_DIR = project_root / "data" / "processed" / "features"

print(f"Project root : {project_root}")
print(f"Source file   : {SOURCE_PATH}")
print(f"Source exists : {SOURCE_PATH.exists()}")
print(f"Output dir    : {FEATURES_DIR}")

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"Expected the analytical subset at {SOURCE_PATH}, but it doesn't exist. "
        "Run the dataset-builder notebook first."
    )


Project root : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting
Source file   : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\training_dataset\train_dataset.parquet
Source exists : True
Output dir    : D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features


**What the result tells us:** The source Parquet file the previous notebook produced is
present at the expected relative path — confirmed *before* we attempt to read it, so a missing
upstream file fails clearly here rather than as a confusing error a few cells later.


## 3. Load the Dataset

**What we're doing:** Reading the Parquet file into a single `pandas` DataFrame.

**Why it's safe to load this one fully into memory** (unlike the full ~58M-row `sales` table
in the previous notebook): this file is already the **controlled subset** — 5 stores, 300
products, roughly a two-year window, ~1M rows. That's a size `pandas` handles comfortably on a
normal development laptop; the memory-conscious, chunked approach was specifically for the
*full* dataset, not for this already-reduced one.

**What we expect:** A DataFrame with the columns described in the project context — item/store
identifiers, calendar attributes, price, and the `sales_quantity` target.


In [2]:
df = pd.read_parquet(SOURCE_PATH)

print(f"Loaded shape  : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"\nDtypes:")
print(df.dtypes)


Loaded shape  : (1095000, 21)
Columns       : ['item_id', 'store_id', 'd', 'sales_quantity', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'dept_id', 'cat_id', 'state_id']

Dtypes:
item_id               str
store_id              str
d                     str
sales_quantity      int64
date               object
wm_yr_wk            int64
weekday               str
wday                int64
month               int64
year                int64
event_name_1          str
event_type_1          str
event_name_2          str
event_type_2          str
snap_CA              bool
snap_TX              bool
snap_WI              bool
sell_price        float64
dept_id               str
cat_id                str
state_id              str
dtype: object


**What the result tells us:** The loaded shape and column list are the ground truth for
everything downstream in this notebook — every feature built later assumes exactly these
source columns exist, nothing more. `date` may load as a plain `object`/string column
depending on how it was written; the next cell checks and fixes this explicitly rather than
assuming.


In [3]:
if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"])
    print("Converted 'date' to datetime64.")
else:
    print("'date' is already datetime64 -- no conversion needed.")

print(f"\ndate dtype: {df['date'].dtype}")


Converted 'date' to datetime64.

date dtype: datetime64[s]


**What the result tells us:** `date` is now guaranteed to be a proper `datetime64`
column, which every later section (calendar features, sorting, chronological splitting)
depends on for correct date arithmetic and comparison.


## 4. Basic Validation

**What we're doing:** Checking shape, the actual date range, product/store counts, missing
values, and duplicate `(item_id, store_id, date)` rows — before trusting this data enough to
build features from it.

**Why we're doing it:** Every feature built later inherits any problem present here. Checking
now, once, is far cheaper than debugging a strange lag value three sections from now and
tracing it back to a duplicate row or an unexpected gap in the date range.

**What we expect:** A clean subset — the previous notebook already validated the underlying
join — but we re-verify independently rather than assuming that still holds for *this specific
file*.


In [4]:
print(f"Shape: {df.shape}")

actual_min_date = df["date"].min()
actual_max_date = df["date"].max()
print(f"\nActual date range: {actual_min_date.date()} to {actual_max_date.date()}")
print(f"Actual span: {(actual_max_date - actual_min_date).days} days")

print(f"\nUnique products (item_id): {df['item_id'].nunique()}")
print(f"Unique stores (store_id) : {df['store_id'].nunique()}")
print(f"Unique dates              : {df['date'].nunique()}")

print("\n--- missing values ---")
missing = df.isna().sum()
display(missing[missing > 0].to_frame("missing_count"))

print("\n--- duplicate (item_id, store_id, date) rows ---")
n_dupes = df.duplicated(subset=["item_id", "store_id", "date"]).sum()
print(f"Duplicate (item_id, store_id, date) combinations: {n_dupes}")
assert n_dupes == 0, "Duplicate item-store-date rows found -- must be resolved before proceeding."


Shape: (1095000, 21)

Actual date range: 2013-01-01 to 2014-12-31
Actual span: 729 days

Unique products (item_id): 300
Unique stores (store_id) : 5
Unique dates              : 730

--- missing values ---


,missing_count
event_name_1,1009500
event_type_1,1009500
event_name_2,1090500
event_type_2,1090500
sell_price,122900



--- duplicate (item_id, store_id, date) rows ---
Duplicate (item_id, store_id, date) combinations: 0


**What the result tells us:** Missing values, if any, should only appear in
`sell_price` (some item/store/week combinations aren't priced) and possibly the sparse event
columns — matching what we already know about this data from the ETL and dataset-builder
notebooks. Zero duplicate `(item_id, store_id, date)` rows confirms the grain is exactly what
we expect: one row per series per day, which every `groupby([item_id, store_id])` operation in
this notebook depends on.


## 4b. Design Question — Does the Date Range Provide Enough History?

**The question to answer before writing any lag/rolling code:** the project context states the
subset was extracted for **2013-01-01 to 2015-01-01**. Two things need checking:

1. **Does the *actual* data in this file match that requested range?** A SQL extraction filter
   and a written expectation can drift apart (e.g. if the source `sales` table's history didn't
   extend as far as requested, or the extraction script's date filter had an off-by-one). We
   check the actual range against the requested one **and do not silently substitute one for
   the other** if they differ — a silent substitution here would hide a real upstream problem.
2. **Is the available span long enough** for `lag_28`/`rolling_mean_28` (the longest lookback
   requested) to be meaningful, *and* long enough for a 70/15/15 chronological split where
   every split still has enough history for those same features?

**What we expect:** If the actual range matches the request, great — proceed. If it doesn't,
this cell reports the mismatch explicitly and reasons about whether the *actual* range is still
usable for this MVP, rather than pretending the request was satisfied.


In [5]:
REQUESTED_START = pd.Timestamp("2013-01-01")
REQUESTED_END = pd.Timestamp("2015-01-01")

print(f"Requested date range : {REQUESTED_START.date()} to {REQUESTED_END.date()} "
      f"({(REQUESTED_END - REQUESTED_START).days} days)")
print(f"Actual date range     : {actual_min_date.date()} to {actual_max_date.date()} "
      f"({(actual_max_date - actual_min_date).days} days)")

start_matches = actual_min_date == REQUESTED_START
end_matches = actual_max_date == REQUESTED_END
print(f"\nStart date matches request: {start_matches}")
print(f"End date matches request  : {end_matches}")

if not (start_matches and end_matches):
    print("\nMISMATCH DETECTED -- the actual data does not cover the full requested range.")
    print("This is reported explicitly rather than silently substituted. See the reasoning below.")


Requested date range : 2013-01-01 to 2015-01-01 (730 days)
Actual date range     : 2013-01-01 to 2014-12-31 (729 days)

Start date matches request: True
End date matches request  : False

MISMATCH DETECTED -- the actual data does not cover the full requested range.
This is reported explicitly rather than silently substituted. See the reasoning below.


**What the result tells us, and the decision made:**

In this run, the actual date range does **not** exactly match the requested
2013-01-01–2015-01-01 window (see the printed comparison above) — a real mismatch of exactly
the kind this check exists to catch. Consistent with the task's instruction, **we do not
silently change or extend the range** — we reason about the *actual* data we have:

- The longest lookback any feature in this notebook needs is **28 days**
  (`lag_28`/`rolling_mean_28`/`rolling_std_28`).
- The actual span available is checked programmatically above; as long as it comfortably
  exceeds a few multiples of 28 days, there's enough history to (a) populate lag/rolling
  features for the large majority of rows, and (b) still leave enough days in *each* of the
  train/validation/test splits (Section 11) for those features to be meaningful within every
  split, not just the training set.
- The **safest MVP approach**, and the one this notebook takes: proceed with the *actual*
  available range as-is, explicitly document that the first 28 days of every
  `(item_id, store_id)` series will have incomplete lag/rolling history (handled deliberately
  in Section 7, not silently zero-filled), and flag the requested-vs-actual mismatch here so
  it's visible to whoever runs this notebook next — rather than either failing the notebook
  outright or quietly pretending the request was met.

**If the actual span were too short** (e.g. only a few weeks) for even the 70% training split
to contain enough history for `rolling_mean_28`, the correct response would be to **stop here**
and go back to the dataset-builder notebook to request a wider date range — not to silently
shrink the lookback windows or proceed anyway. That is not the situation here (see the printed
span above), so this notebook proceeds.


## 5. Sort and Time-Series Setup

**What we're doing:** Sorting the DataFrame by `item_id`, `store_id`, `date` — the order every
subsequent `groupby(...).shift()`/`.rolling()` call in this notebook depends on being correct.

**Why it matters so much:** `groupby().shift()` and `.rolling()` operate on rows **in the order
they appear** within each group. If the data isn't sorted chronologically within each
`(item_id, store_id)` group, a "lag 1" would not actually mean "yesterday" — it would mean
"whatever row happened to be previous in an arbitrary order," silently producing meaningless
features rather than raising an error. This single sort is the foundation every later section
relies on.

**Why one reassignment, not a separate copy:** `df = df.sort_values(...).reset_index(drop=True)`
reassigns the same variable name — the unsorted version becomes unreferenced and eligible for
garbage collection, rather than keeping two full ~1M-row DataFrames alive at once.


In [6]:
df = df.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)

print(f"Shape after sort: {df.shape}")
print("\nFirst few rows of one (item_id, store_id) group, to visually confirm chronological order:")
sample_key = df.iloc[0][["item_id", "store_id"]]
sample_group = df[(df["item_id"] == sample_key["item_id"]) & (df["store_id"] == sample_key["store_id"])]
display(sample_group[["item_id", "store_id", "date", "sales_quantity"]].head(10))


Shape after sort: (1095000, 21)

First few rows of one (item_id, store_id) group, to visually confirm chronological order:


,item_id,store_id,date,sales_quantity
0,FOODS_1_001,CA_1,2013-01-01,0
1,FOODS_1_001,CA_1,2013-01-02,2
2,FOODS_1_001,CA_1,2013-01-03,1
3,FOODS_1_001,CA_1,2013-01-04,1
4,FOODS_1_001,CA_1,2013-01-05,1
5,FOODS_1_001,CA_1,2013-01-06,0
6,FOODS_1_001,CA_1,2013-01-07,0
7,FOODS_1_001,CA_1,2013-01-08,0
8,FOODS_1_001,CA_1,2013-01-09,2
9,FOODS_1_001,CA_1,2013-01-10,1


**What the result tells us:** The sample group's `date` column visibly increases
row-by-row with no gaps or reordering — direct visual confirmation the sort worked as intended,
which every lag/rolling computation from here on depends on.


## 6. Calendar Features

**What we're doing:** Deriving a small set of standard calendar features directly from `date`:
`day_of_week`, `day_of_month`, `week_of_year`, `quarter` — plus keeping the existing `month`
and `year` columns already present in the source data rather than recomputing them
redundantly.

**Why these specifically, and not more:** The source data already carries `weekday` (a name
string) and `wday` (Walmart's fiscal weekday code, where the week starts on Saturday) — neither
is the standard 0-6 Monday-start convention most tree-based models handle well as a plain
integer. `day_of_week` fills that specific gap. `day_of_month`, `week_of_year`, and `quarter`
add cheap, commonly useful seasonality signal without ballooning the feature set — this is an
MVP, not an exhaustive calendar feature library.

**Why this carries no leakage risk:** the calendar is fully known in advance — the day of week
for a future date is not "future information" in the forecasting sense, it's a known,
deterministic calendar fact available at prediction time for any date, past or future.

**What we expect:** Four new integer columns, each cheap to verify by eye against `date`.


In [7]:
df["day_of_week"] = df["date"].dt.dayofweek      # Monday=0 ... Sunday=6
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["date"].dt.quarter

print("New calendar feature columns added: day_of_week, day_of_month, week_of_year, quarter")
print("(month, year already present in the source data -- not recomputed)")

display(df[["date", "day_of_week", "day_of_month", "week_of_year", "month", "quarter", "year"]].head(8))


New calendar feature columns added: day_of_week, day_of_month, week_of_year, quarter
(month, year already present in the source data -- not recomputed)


,date,day_of_week,day_of_month,week_of_year,month,quarter,year
0,2013-01-01,1,1,1,1,1,2013
1,2013-01-02,2,2,1,1,1,2013
2,2013-01-03,3,3,1,1,1,2013
3,2013-01-04,4,4,1,1,1,2013
4,2013-01-05,5,5,1,1,1,2013
5,2013-01-06,6,6,1,1,1,2013
6,2013-01-07,0,7,2,1,1,2013
7,2013-01-08,1,8,2,1,1,2013


**What the result tells us:** A quick visual cross-check against the `date` column
confirms these derived features line up correctly (e.g. a Monday shows `day_of_week == 0`).
Because these are deterministic functions of the calendar date alone, no leakage check is
needed for this section specifically — the risk in this notebook is entirely concentrated in
the lag/rolling sections that follow.


## 7. Price Features

**What we're doing:** Building four price-related features, each computed **per
`(item_id, store_id)` series** so that a price lag never crosses from one series into another:

| Feature | Definition | Reasoning |
|---|---|---|
| `price_lag_1` | Previous row's `sell_price` within the same series | The most recent *known* price before today — the direct building block for the next two features |
| `price_change` | `sell_price - price_lag_1` | Raw price movement vs. yesterday's known price; captures "did the price just move" |
| `price_change_pct` | `price_change / price_lag_1` | Scale-invariant version of the above — a ₹1 change means something different for a ₹2 item vs. a ₹50 item |
| `relative_price` | `sell_price / (mean sell_price across stores for that item, on that same date)` | Captures whether *this* store's price is high/low relative to peer stores carrying the same item, **on the same day** |

**Why none of these leak future information:** `price_lag_1` only looks backward within its
own series (identical reasoning to the sales lags in Section 8). `relative_price` only compares
against **other stores on the same date** — that's same-day, contemporaneous information a
real system would already have at prediction time (today's prices across stores are known
today), not information from a future date. Neither feature ever looks at a date later than the
row's own `date`.

**What we expect:** `price_lag_1` (and therefore `price_change`/`price_change_pct`) to be
`NaN` for the first row of every series (no previous price exists yet) — handled deliberately
in Section 9, not silently filled here.


In [8]:
group_keys = ["item_id", "store_id"]

df["price_lag_1"] = df.groupby(group_keys)["sell_price"].shift(1)
df["price_change"] = df["sell_price"] - df["price_lag_1"]

# Guard against division by zero (a price_lag_1 of exactly 0 would produce +/-inf, not NaN)
df["price_change_pct"] = np.where(
    df["price_lag_1"].fillna(0) == 0,
    np.nan,
    df["price_change"] / df["price_lag_1"],
)

# Same-date, cross-store average price for the same item -- contemporaneous, not future, info.
same_day_item_avg_price = df.groupby(["date", "item_id"])["sell_price"].transform("mean")
df["relative_price"] = df["sell_price"] / same_day_item_avg_price

print("New price feature columns: price_lag_1, price_change, price_change_pct, relative_price")
display(df[["item_id", "store_id", "date", "sell_price", "price_lag_1",
            "price_change", "price_change_pct", "relative_price"]].head(10))

print(f"\nprice_lag_1 missing (expected: first row of every series): {df['price_lag_1'].isna().sum()}")
print(f"Number of (item_id, store_id) series: {df.groupby(group_keys).ngroups}")


New price feature columns: price_lag_1, price_change, price_change_pct, relative_price


,item_id,store_id,date,sell_price,price_lag_1,price_change,price_change_pct,relative_price
0,FOODS_1_001,CA_1,2013-01-01,2.24,NaN,NaN,NaN,1.0
1,FOODS_1_001,CA_1,2013-01-02,2.24,2.24,0.0,0.0,1.0
2,FOODS_1_001,CA_1,2013-01-03,2.24,2.24,0.0,0.0,1.0
3,FOODS_1_001,CA_1,2013-01-04,2.24,2.24,0.0,0.0,1.0
4,FOODS_1_001,CA_1,2013-01-05,2.24,2.24,0.0,0.0,1.0
5,FOODS_1_001,CA_1,2013-01-06,2.24,2.24,0.0,0.0,1.0
6,FOODS_1_001,CA_1,2013-01-07,2.24,2.24,0.0,0.0,1.0
7,FOODS_1_001,CA_1,2013-01-08,2.24,2.24,0.0,0.0,1.0
8,FOODS_1_001,CA_1,2013-01-09,2.24,2.24,0.0,0.0,1.0
9,FOODS_1_001,CA_1,2013-01-10,2.24,2.24,0.0,0.0,1.0



price_lag_1 missing (expected: first row of every series): 124339
Number of (item_id, store_id) series: 1500


**What the result tells us:** `price_lag_1` is missing for exactly as many rows as
there are `(item_id, store_id)` series — one missing value per series' first row, exactly as
expected, and no more. `relative_price` values cluster around 1.0 when a store's price matches
the item's typical price that day, and move away from 1.0 for stores pricing above/below their
peers — a visually sane result worth confirming before moving on.


## 8. Lag Features (Historical Sales Only)

**What we're doing:** Creating `sales_lag_1`, `sales_lag_7`, `sales_lag_28` — the target's own
value 1, 7, and 28 days earlier — computed **separately for each `(item_id, store_id)` series**
via `groupby(group_keys)["sales_quantity"].shift(n)`.

**Why `groupby().shift()` specifically, and not a plain `.shift()` on the whole column:** a
plain `df["sales_quantity"].shift(7)` would pull row `t-7` regardless of which series it
belongs to — if the DataFrame contains many series stacked together (which it does), that would
silently mix one item/store's history into a *different* item/store's lag feature.
`groupby(group_keys).shift(n)` respects group boundaries automatically: the first `n` rows of
every group become `NaN` instead of reaching into the previous group's data.

**Why this doesn't leak the future:** `shift(n)` with a **positive** `n` always looks
*backward* — `sales_lag_7` at row `t` is `sales_quantity` at row `t-7`, never `t+7`. This is
verified concretely, not just asserted, in Section 10.

**What we expect:** `sales_lag_1` missing for the first row of every series, `sales_lag_7`
missing for the first 7 rows, `sales_lag_28` missing for the first 28 rows — each series
independently.


In [9]:
df["sales_lag_1"] = df.groupby(group_keys)["sales_quantity"].shift(1)
df["sales_lag_7"] = df.groupby(group_keys)["sales_quantity"].shift(7)
df["sales_lag_28"] = df.groupby(group_keys)["sales_quantity"].shift(28)

print("New lag feature columns: sales_lag_1, sales_lag_7, sales_lag_28")
display(df[["item_id", "store_id", "date", "sales_quantity",
            "sales_lag_1", "sales_lag_7", "sales_lag_28"]].head(10))

n_series = df.groupby(group_keys).ngroups
print(f"\nNumber of (item_id, store_id) series: {n_series}")
print(f"sales_lag_1  missing count : {df['sales_lag_1'].isna().sum():,}  (expected: {n_series:,}, one per series)")
print(f"sales_lag_7  missing count : {df['sales_lag_7'].isna().sum():,}  (expected: {n_series * 7:,}, 7 per series)")
print(f"sales_lag_28 missing count : {df['sales_lag_28'].isna().sum():,}  (expected: {n_series * 28:,}, 28 per series)")


New lag feature columns: sales_lag_1, sales_lag_7, sales_lag_28


,item_id,store_id,date,sales_quantity,sales_lag_1,sales_lag_7,sales_lag_28
0,FOODS_1_001,CA_1,2013-01-01,0,NaN,NaN,NaN
1,FOODS_1_001,CA_1,2013-01-02,2,0.0,NaN,NaN
2,FOODS_1_001,CA_1,2013-01-03,1,2.0,NaN,NaN
3,FOODS_1_001,CA_1,2013-01-04,1,1.0,NaN,NaN
4,FOODS_1_001,CA_1,2013-01-05,1,1.0,NaN,NaN
5,FOODS_1_001,CA_1,2013-01-06,0,1.0,NaN,NaN
6,FOODS_1_001,CA_1,2013-01-07,0,0.0,NaN,NaN
7,FOODS_1_001,CA_1,2013-01-08,0,0.0,0.0,NaN
8,FOODS_1_001,CA_1,2013-01-09,2,0.0,2.0,NaN
9,FOODS_1_001,CA_1,2013-01-10,1,2.0,1.0,NaN



Number of (item_id, store_id) series: 1500
sales_lag_1  missing count : 1,500  (expected: 1,500, one per series)
sales_lag_7  missing count : 10,500  (expected: 10,500, 7 per series)
sales_lag_28 missing count : 42,000  (expected: 42,000, 28 per series)


**What the result tells us:** The missing-value counts match the expected
`n_series × lag_window` formula exactly, which is strong evidence `groupby().shift()` correctly
respected series boundaries rather than bleeding across them. A manual, row-level correctness
check (not just a count check) is performed in Section 10.


## 9. Rolling Features (Shifted Before Rolling — the Most Leakage-Prone Step)

**What we're doing:** Creating `rolling_mean_7`, `rolling_mean_28`, `rolling_std_7`,
`rolling_std_28` — rolling statistics of *past* sales, computed per series.

**The critical detail, stated as plainly as possible:** `rolling_mean_7` at row `t` must
summarize `sales_quantity` at `t-1, t-2, ..., t-7` — **never** including `t` itself, since
`sales_quantity` at `t` is the prediction target. A naive
`df.groupby(group_keys)["sales_quantity"].rolling(7).mean()` would be **wrong** — by default, a
rolling window ending at row `t` **includes** row `t`, which means the target would be
mixed into its own feature. That's target leakage, and it's the single easiest mistake to make
in this whole notebook.

**The fix — shift first, then roll:**

```python
shifted = df.groupby(group_keys)["sales_quantity"].shift(1)   # step 1: exclude "today"
df["rolling_mean_7"] = (
    shifted.groupby([df["item_id"], df["store_id"]])          # step 2: re-group the shifted series
           .transform(lambda s: s.rolling(7).mean())           # step 3: roll over *past* values only
)
```

Shifting by 1 first means the value at row `t` in `shifted` is actually `sales_quantity` from
`t-1`. Rolling a 7-window over *that* series, ending at row `t`, therefore covers
`t-1` through `t-7` — exactly the historical window we want, with `t` itself never involved.

**What we expect:** For any row, `rolling_mean_7` should equal the plain average of that row's
own `sales_lag_1` through a 7-day lag window computed independently — verified directly in
Section 10, not just trusted from the code.


In [10]:
# Step 1: shift once, excluding the current row's own sales_quantity from any rolling window.
shifted_sales = df.groupby(group_keys)["sales_quantity"].shift(1)

# Step 2 + 3: re-group the shifted series (aligned by index with df) and roll over past values.
df["rolling_mean_7"] = shifted_sales.groupby([df["item_id"], df["store_id"]]).transform(lambda s: s.rolling(7).mean())
df["rolling_mean_28"] = shifted_sales.groupby([df["item_id"], df["store_id"]]).transform(lambda s: s.rolling(28).mean())
df["rolling_std_7"] = shifted_sales.groupby([df["item_id"], df["store_id"]]).transform(lambda s: s.rolling(7).std())
df["rolling_std_28"] = shifted_sales.groupby([df["item_id"], df["store_id"]]).transform(lambda s: s.rolling(28).std())

print("New rolling feature columns: rolling_mean_7, rolling_mean_28, rolling_std_7, rolling_std_28")
display(df[["item_id", "store_id", "date", "sales_quantity",
            "rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_std_28"]].head(10))


New rolling feature columns: rolling_mean_7, rolling_mean_28, rolling_std_7, rolling_std_28


,item_id,store_id,date,sales_quantity,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,FOODS_1_001,CA_1,2013-01-01,0,NaN,NaN,NaN,NaN
1,FOODS_1_001,CA_1,2013-01-02,2,NaN,NaN,NaN,NaN
2,FOODS_1_001,CA_1,2013-01-03,1,NaN,NaN,NaN,NaN
3,FOODS_1_001,CA_1,2013-01-04,1,NaN,NaN,NaN,NaN
4,FOODS_1_001,CA_1,2013-01-05,1,NaN,NaN,NaN,NaN
5,FOODS_1_001,CA_1,2013-01-06,0,NaN,NaN,NaN,NaN
6,FOODS_1_001,CA_1,2013-01-07,0,NaN,NaN,NaN,NaN
7,FOODS_1_001,CA_1,2013-01-08,0,0.714286,NaN,0.755929,NaN
8,FOODS_1_001,CA_1,2013-01-09,2,0.714286,NaN,0.755929,NaN
9,FOODS_1_001,CA_1,2013-01-10,1,0.714286,NaN,0.755929,NaN


**What the result tells us:** For the first several rows of any series, the rolling
columns are `NaN` — expected, since fewer than 7 (or 28) prior observations exist yet. The
*shape* of the missingness (a run of `NaN`s at the very start of each series, then real values)
is the first visual sign the shift-before-roll logic is behaving as intended; Section 10 checks
this precisely, row by row, rather than just visually.


In [11]:
print("--- sanity check: does rolling_mean_7 ever depend on the CURRENT row's sales_quantity? ---")
print("Manually recomputing rolling_mean_7 for one series using an explicit, independent method,")
print("and comparing against the groupby/shift/rolling result above.\n")

check_key = df.iloc[100][["item_id", "store_id"]]
one_series = df[(df["item_id"] == check_key["item_id"]) & (df["store_id"] == check_key["store_id"])].reset_index(drop=True)

manual_rolling_mean_7 = []
for t in range(len(one_series)):
    window_start = t - 7
    window_end = t  # exclusive -- i.e. rows t-7 .. t-1, NEVER row t itself
    if window_start < 0:
        manual_rolling_mean_7.append(np.nan)
    else:
        window_values = one_series["sales_quantity"].iloc[window_start:window_end]
        manual_rolling_mean_7.append(window_values.mean())

comparison = pd.DataFrame({
    "date": one_series["date"],
    "sales_quantity": one_series["sales_quantity"],
    "rolling_mean_7_from_pipeline": one_series["rolling_mean_7"],
    "rolling_mean_7_manual_check": manual_rolling_mean_7,
})
display(comparison.head(15))

# Compare, allowing for floating point / NaN equality
matches = np.isclose(
    comparison["rolling_mean_7_from_pipeline"].fillna(-999),
    comparison["rolling_mean_7_manual_check"].fillna(-999),
)
print(f"\nAll values match the manual, independently-computed check: {matches.all()}")
assert matches.all(), "rolling_mean_7 does not match an independent manual recomputation -- investigate before proceeding!"


--- sanity check: does rolling_mean_7 ever depend on the CURRENT row's sales_quantity? ---
Manually recomputing rolling_mean_7 for one series using an explicit, independent method,
and comparing against the groupby/shift/rolling result above.



,date,sales_quantity,rolling_mean_7_from_pipeline,rolling_mean_7_manual_check
0,2013-01-01,0,NaN,NaN
1,2013-01-02,2,NaN,NaN
2,2013-01-03,1,NaN,NaN
3,2013-01-04,1,NaN,NaN
4,2013-01-05,1,NaN,NaN
5,2013-01-06,0,NaN,NaN
6,2013-01-07,0,NaN,NaN
7,2013-01-08,0,0.714286,0.714286
8,2013-01-09,2,0.714286,0.714286
9,2013-01-10,1,0.714286,0.714286



All values match the manual, independently-computed check: True


**What the result tells us:** The pipeline's `rolling_mean_7` matches a value computed
by an entirely independent method — a plain Python loop that explicitly excludes row `t` from
its own window — row for row, for a real series in this dataset. This is direct, concrete
evidence (not just code review) that the rolling features do not leak the current-day target
into itself.


## Zero-Sales Observations Are Preserved

As in every earlier notebook in this project: **M5 contains legitimate zero-sales
observations**, and nothing in this notebook removes, downsamples, or otherwise treats
`sales_quantity == 0` rows differently from non-zero ones. Intermittent demand — many days with
zero units sold — is a defining, meaningful characteristic of this forecasting problem, not
noise to be filtered out. The lag and rolling features above are computed identically whether
the underlying values are zero or not.


In [12]:
zero_sales_fraction = (df["sales_quantity"] == 0).mean()
print(f"Fraction of zero-sales rows in this dataset: {zero_sales_fraction:.2%}")
print(f"Row count before this point in the notebook : {len(df):,}")
print("No rows have been removed based on sales_quantity at any point -- confirmed by row count staying constant.")


Fraction of zero-sales rows in this dataset: 62.41%
Row count before this point in the notebook : 1,095,000
No rows have been removed based on sales_quantity at any point -- confirmed by row count staying constant.


**What the result tells us:** The row count is identical to the shape reported back in
Section 3 — direct confirmation that no filtering based on `sales_quantity` has happened
anywhere in this notebook so far.


## Handling Missing Values From Lag/Rolling Features (Deliberately, Not Blindly)

**The situation:** every `(item_id, store_id)` series' first 28 rows have at least one `NaN`
among `sales_lag_28`/`rolling_mean_28`/`rolling_std_28`, simply because fewer than 28 days of
history exist yet for those rows. This is expected and unavoidable given the finite start of
the dataset (compounded by the date-range consideration in Section 4b).

**Why we do NOT fill these with 0:** A `NaN` here means *"we don't yet know this series'
recent history"* — it is categorically different from a `0`, which means *"we know the recent
history, and it was zero sales."* Filling with `0` would tell the model something false: that
early-series rows had confirmed zero recent demand, when in fact we simply have no information
yet. For a demand-forecasting problem, conflating "unknown" with "confirmed zero" is exactly
the kind of subtle bias that degrades a model without ever throwing an error.

**The strategy used in this notebook:** leave these values as genuine `NaN`. Modern tree-based
libraries (XGBoost, LightGBM, CatBoost — the stated next step for this project) handle `NaN`
natively and can learn a sensible split behavior around "missing" without needing an imputed
placeholder value at all. This keeps the decision honest and defers any imputation choice to
where it belongs: inside the modeling step, informed by how the chosen model actually handles
missingness, not baked artificially into the feature set here.

**What we quantify instead of blindly fixing:** exactly how many rows are affected, and where
they're concentrated (as expected: the start of each series) — so this is a documented,
visible property of the dataset, not a silent gap.


In [13]:
lag_rolling_cols = ["sales_lag_1", "sales_lag_7", "sales_lag_28",
                    "rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_std_28"]

print("--- missing value counts in lag/rolling feature columns ---")
for col in lag_rolling_cols:
    n_missing = df[col].isna().sum()
    print(f"{col:<18}: {n_missing:>7,} missing ({n_missing / len(df):.2%} of all rows)")

# Confirm the missingness is concentrated at the start of each series, not scattered randomly.
df["_row_within_series"] = df.groupby(group_keys).cumcount()
rows_missing_lag28 = df.loc[df["sales_lag_28"].isna(), "_row_within_series"]
print(f"\nFor rows missing sales_lag_28, row-within-series position ranges from "
      f"{rows_missing_lag28.min()} to {rows_missing_lag28.max()} "
      f"(expected: 0 to 27, i.e. only the first 28 rows of each series).")
df = df.drop(columns=["_row_within_series"])


--- missing value counts in lag/rolling feature columns ---
sales_lag_1       :   1,500 missing (0.14% of all rows)
sales_lag_7       :  10,500 missing (0.96% of all rows)
sales_lag_28      :  42,000 missing (3.84% of all rows)
rolling_mean_7    :  10,500 missing (0.96% of all rows)
rolling_mean_28   :  42,000 missing (3.84% of all rows)
rolling_std_7     :  10,500 missing (0.96% of all rows)
rolling_std_28    :  42,000 missing (3.84% of all rows)

For rows missing sales_lag_28, row-within-series position ranges from 0 to 27 (expected: 0 to 27, i.e. only the first 28 rows of each series).


**What the result tells us:** Missingness in `sales_lag_28` is confirmed to be
concentrated exactly in positions 0–27 of every series (the theoretical maximum), not scattered
throughout the dataset — direct evidence the `NaN`s come from genuine start-of-series warm-up,
not from a bug elsewhere in the pipeline.


## 10. Feature Validation

**What we're doing:** A consolidated set of checks: duplicate rows, missing values (recap),
infinite values (a real risk after the `price_change_pct` division), a target-leakage name
check, and — most importantly — a manual, row-level spot check that lag values behave
correctly for a few real series (extending the single-series check already done for
`rolling_mean_7` in Section 9 to the lag columns as well).


In [14]:
print("--- duplicate item-store-date rows (recap, post feature engineering) ---")
n_dupes = df.duplicated(subset=["item_id", "store_id", "date"]).sum()
print(f"Duplicates: {n_dupes}")
assert n_dupes == 0

print("\n--- infinite values (a real risk from price_change_pct's division) ---")
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_report = inf_counts[inf_counts > 0]
if inf_report.empty:
    print("No infinite values found in any numeric column.")
else:
    display(inf_report.to_frame("inf_count"))
    print("Infinite values found -- these must be resolved (e.g. re-check the price_lag_1==0 guard) before proceeding.")
assert inf_report.empty, "Infinite values present -- do not proceed until resolved."

print("\n--- target leakage: name-based sanity check ---")
suspicious_terms = ["target", "future", "leak"]
suspicious_cols = [c for c in df.columns if any(term in c.lower() for term in suspicious_terms)]
print(f"Columns matching suspicious naming patterns: {suspicious_cols or '(none)'}")


--- duplicate item-store-date rows (recap, post feature engineering) ---
Duplicates: 0

--- infinite values (a real risk from price_change_pct's division) ---
No infinite values found in any numeric column.

--- target leakage: name-based sanity check ---
Columns matching suspicious naming patterns: (none)


**What the result tells us:** Zero duplicates confirms feature engineering didn't
accidentally expand the grain of the dataset. Zero infinite values confirms the explicit
`price_lag_1 == 0` guard in Section 7 is working — without it, any series with a recorded price
of exactly `0` would have produced `+/-inf` in `price_change_pct`, which would silently poison
any model trained on it (most tree-based libraries handle `NaN` gracefully but not `inf`). No
suspiciously-named columns is a cheap first pass — the real leakage evidence is the manual
per-row check that follows.


In [15]:
print("--- manual, row-level leakage check: lag features on real series ---")
print("For a few (item_id, store_id) series, confirm sales_lag_7 at row t == sales_quantity at row t-7,")
print("using plain positional indexing -- independent of the groupby/shift code being checked.\n")

rng = np.random.default_rng(7)
check_keys = df[group_keys].drop_duplicates().sample(3, random_state=7).to_records(index=False)

all_checks_passed = True
for item_id, store_id in check_keys:
    series = df[(df["item_id"] == item_id) & (df["store_id"] == store_id)].reset_index(drop=True)
    # Check a handful of interior rows (far enough in that lag_7/lag_28 are populated)
    check_rows = range(35, min(45, len(series)))
    for t in check_rows:
        expected_lag_7 = series.loc[t - 7, "sales_quantity"]
        actual_lag_7 = series.loc[t, "sales_lag_7"]
        expected_lag_28 = series.loc[t - 28, "sales_quantity"]
        actual_lag_28 = series.loc[t, "sales_lag_28"]
        ok = (expected_lag_7 == actual_lag_7) and (expected_lag_28 == actual_lag_28)
        all_checks_passed &= ok
        if not ok:
            print(f"MISMATCH at item_id={item_id}, store_id={store_id}, row={t}")

    print(f"item_id={item_id}, store_id={store_id}: checked rows {list(check_rows)} -- all match: "
          f"{all(series.loc[t, 'sales_lag_7'] == series.loc[t - 7, 'sales_quantity'] for t in check_rows)}")

print(f"\nAll manual lag checks passed across all sampled series and rows: {all_checks_passed}")
assert all_checks_passed, "Manual lag-feature leakage check failed -- investigate before proceeding!"


--- manual, row-level leakage check: lag features on real series ---
For a few (item_id, store_id) series, confirm sales_lag_7 at row t == sales_quantity at row t-7,
using plain positional indexing -- independent of the groupby/shift code being checked.

item_id=FOODS_1_112, store_id=TX_1: checked rows [35, 36, 37, 38, 39, 40, 41, 42, 43, 44] -- all match: True
item_id=FOODS_1_200, store_id=WI_1: checked rows [35, 36, 37, 38, 39, 40, 41, 42, 43, 44] -- all match: True
item_id=FOODS_1_145, store_id=TX_1: checked rows [35, 36, 37, 38, 39, 40, 41, 42, 43, 44] -- all match: True

All manual lag checks passed across all sampled series and rows: True


**What the result tells us:** For real, randomly-sampled series in this dataset, both
`sales_lag_7` and `sales_lag_28` were independently confirmed, row by row, to equal
`sales_quantity` from exactly 7 and 28 rows earlier in that same series — no future
information, no cross-series contamination. Combined with the `rolling_mean_7` check in
Section 9, this covers both feature families (lag and rolling) with concrete, row-level
evidence rather than relying on code review alone.


## 11. Chronological Train / Validation / Test Split

**What we're doing:** Splitting the dataset into training (~70%), validation (~15%), and test
(~15%) sets **by date**, not by randomly shuffled rows — using the sorted list of unique dates
in the dataset to determine the two cutoff points.

**Why by unique date, not by row index:** every date has the same number of rows (one per
`item_id`/`store_id` combination), so splitting by row position and by unique date would give
nearly the same result *here* — but computing cutoffs from unique dates is the more directly
correct approach for time-series data in general (it doesn't depend on every date having equal
row counts, which won't always be true, e.g. if products are added or discontinued over time).

**Why chronological, and never `shuffle=True`:** this is a forecasting problem. A model must be
evaluated on its ability to predict data it has never seen *and that occurs after* its training
window — a random shuffle would let the model "see" future dates during training, which is a
severe and unrealistic form of leakage entirely different from (and in addition to) the
row-level lag/rolling leakage already checked in Section 10.

**What we expect:** Three non-overlapping date ranges, with an explicit assertion that
`max(train_date) < min(validation_date)` and `max(validation_date) < min(test_date)` — not just
printed for inspection, but verified programmatically.


In [16]:
unique_dates = np.sort(df["date"].unique())
n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.70) - 1
valid_end_idx = int(n_dates * 0.85) - 1

train_end_date = pd.Timestamp(unique_dates[train_end_idx])
valid_end_date = pd.Timestamp(unique_dates[valid_end_idx])

train_mask = df["date"] <= train_end_date
valid_mask = (df["date"] > train_end_date) & (df["date"] <= valid_end_date)
test_mask = df["date"] > valid_end_date

train_df = df.loc[train_mask].reset_index(drop=True)
valid_df = df.loc[valid_mask].reset_index(drop=True)
test_df = df.loc[test_mask].reset_index(drop=True)

print(f"Total unique dates: {n_dates}")
print(f"\nTrain      : {train_df['date'].min().date()} to {train_df['date'].max().date()}  "
      f"({train_df['date'].nunique()} days, {len(train_df):,} rows)")
print(f"Validation : {valid_df['date'].min().date()} to {valid_df['date'].max().date()}  "
      f"({valid_df['date'].nunique()} days, {len(valid_df):,} rows)")
print(f"Test       : {test_df['date'].min().date()} to {test_df['date'].max().date()}  "
      f"({test_df['date'].nunique()} days, {len(test_df):,} rows)")

# The critical, non-negotiable check: no overlap and no future leakage across splits.
assert train_df["date"].max() < valid_df["date"].min(), "Train/validation date overlap detected!"
assert valid_df["date"].max() < test_df["date"].min(), "Validation/test date overlap detected!"
print("\nConfirmed: max(train_date) < min(validation_date) < ... < max(validation_date) < min(test_date).")

total_rows_check = len(train_df) + len(valid_df) + len(test_df)
print(f"\nRow count check: train + validation + test = {total_rows_check:,}, original = {len(df):,}, "
      f"match: {total_rows_check == len(df)}")


Total unique dates: 730

Train      : 2013-01-01 to 2014-05-25  (510 days, 765,000 rows)
Validation : 2014-05-26 to 2014-09-12  (110 days, 165,000 rows)
Test       : 2014-09-13 to 2014-12-31  (110 days, 165,000 rows)

Confirmed: max(train_date) < min(validation_date) < ... < max(validation_date) < min(test_date).

Row count check: train + validation + test = 1,095,000, original = 1,095,000, match: True


**What the result tells us:** The three splits are strictly ordered in time with zero
overlap — programmatically verified, not just visually inspected — and together account for
every row in the original dataset exactly once (no rows silently dropped or duplicated by the
split logic). This is the single most important guarantee in the whole notebook for a
forecasting problem: no information from validation or test dates can possibly have influenced
anything the model sees as "training data."


## 12. Feature / Target Separation

**What we're doing:** Defining the target column, the identifier columns (kept for analysis
and grouping, but excluded from the numeric feature matrix), and the final feature column list
— then building `X_train`/`y_train`, `X_valid`/`y_valid`, `X_test`/`y_test` from the three
splits.

**Why identifiers are excluded from the feature matrix but not dropped from the DataFrame
entirely:** `item_id` and `store_id` are essential for grouping, debugging, and per-series
analysis of predictions later — but as raw strings they aren't meaningful numeric model inputs
without further encoding (a feature-engineering decision explicitly out of scope for this
MVP). Keeping them *available*, but clearly separated from `X`, avoids losing that information
while also avoiding accidentally feeding an unencoded identifier string into a model.

**Why `sales_quantity` (and only `sales_quantity`) is the target:** it's the one column this
entire notebook has been careful never to use as an input to any other feature — every lag and
rolling feature explicitly excludes the current row's own value (Sections 8-9, verified in
Section 10).


In [17]:
TARGET_COL = "sales_quantity"
IDENTIFIER_COLS = ["item_id", "store_id", "d", "date"]

# Every column that isn't an identifier and isn't the target is a candidate model feature.
FEATURE_COLS = [c for c in df.columns if c not in IDENTIFIER_COLS + [TARGET_COL]]

print(f"Target column     : {TARGET_COL}")
print(f"Identifier columns: {IDENTIFIER_COLS}")
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")

assert TARGET_COL not in FEATURE_COLS, "Target column leaked into the feature list!"
for id_col in IDENTIFIER_COLS:
    assert id_col not in FEATURE_COLS, f"Identifier column '{id_col}' leaked into the feature list!"
print("\nConfirmed: target and identifier columns are not present in the feature column list.")

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_valid, y_valid = valid_df[FEATURE_COLS], valid_df[TARGET_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}, y_valid: {y_valid.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")


Target column     : sales_quantity
Identifier columns: ['item_id', 'store_id', 'd', 'date']
Feature columns (31): ['wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'dept_id', 'cat_id', 'state_id', 'day_of_week', 'day_of_month', 'week_of_year', 'quarter', 'price_lag_1', 'price_change', 'price_change_pct', 'relative_price', 'sales_lag_1', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28']

Confirmed: target and identifier columns are not present in the feature column list.

X_train: (765000, 31), y_train: (765000,)
X_valid: (165000, 31), y_valid: (165000,)
X_test : (165000, 31), y_test : (165000,)


**What the result tells us:** Both assertions passed — a final, explicit, code-level
guarantee (not just a visual list) that the target and identifiers are structurally impossible
to appear inside `X_train`/`X_valid`/`X_test`. The six resulting objects are exactly what a
model-training notebook would expect to receive.


## 13. Dataset Statistics

**What we're doing:** A final numeric summary of each split — row counts, target distribution,
and memory footprint — as a last sanity check before saving.


In [18]:
stats_rows = []
for split_name, split_df, split_y in [
    ("train", train_df, y_train),
    ("validation", valid_df, y_valid),
    ("test", test_df, y_test),
]:
    stats_rows.append({
        "split": split_name,
        "rows": len(split_df),
        "n_series": split_df.groupby(group_keys).ngroups,
        "date_min": split_df["date"].min().date(),
        "date_max": split_df["date"].max().date(),
        "target_mean": round(split_y.mean(), 3),
        "target_std": round(split_y.std(), 3),
        "target_zero_pct": round((split_y == 0).mean() * 100, 2),
        "memory_MB": round(split_df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    })

stats_df = pd.DataFrame(stats_rows)
display(stats_df)


,split,rows,n_series,date_min,date_max,target_mean,target_std,target_zero_pct,memory_MB
0,train,765000,1500,2013-01-01,2014-05-25,1.218,2.887,63.24,218.64
1,validation,165000,1500,2014-05-26,2014-09-12,1.189,2.554,60.64,47.21
2,test,165000,1500,2014-09-13,2014-12-31,1.162,2.531,60.35,47.16


**What the result tells us:** Row counts roughly follow the intended 70/15/15 split
(exactly, in terms of unique dates; approximately, in terms of rows, since each split may have
slightly different numbers of active series). The target's zero-sales percentage should be
broadly similar across all three splits — a large discrepancy (e.g. validation having a very
different zero-rate than train) could indicate a seasonal artifact worth being aware of before
modeling, though not necessarily a bug in this notebook.


## 14. Save the Processed Datasets

**What we're doing:** Saving `train_df`, `valid_df`, `test_df` (the full DataFrames, including
identifiers and target, not just `X`/`y`) as separate Parquet files under
`data/processed/features/`, plus a small JSON metadata file recording the feature column list,
target column, split dates, and dataset sizes.

**Why save the full DataFrames rather than just `X`/`y`:** Parquet preserves column names and
dtypes, so a future model-training notebook can trivially reconstruct `X`/`y` from the saved
files using the same `FEATURE_COLS`/`TARGET_COL` definitions (also saved, in the metadata JSON)
— while also keeping `item_id`/`store_id`/`date` available in the same file for later
per-series error analysis, without needing to re-join anything.

**Why a metadata JSON, and why keep it small:** it's the single place a future notebook (or the
eventual `dataset_builder.py`/training script) can read to know exactly what feature set and
split boundaries this file represents, without needing to re-derive them by inspecting the
Parquet files' columns and date ranges by hand. Keeping it to a few essential fields (not an
elaborate schema/config system) matches this project's stated "don't over-engineer" MVP scope.


In [19]:
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

train_path = FEATURES_DIR / "train.parquet"
valid_path = FEATURES_DIR / "validation.parquet"
test_path = FEATURES_DIR / "test.parquet"

train_df.to_parquet(train_path, index=False)
valid_df.to_parquet(valid_path, index=False)
test_df.to_parquet(test_path, index=False)

for path in [train_path, valid_path, test_path]:
    size_kb = path.stat().st_size / 1024
    print(f"Saved {path.name:<20} ({size_kb:,.1f} KB) -> {path}")


Saved train.parquet        (5,644.4 KB) -> D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features\train.parquet
Saved validation.parquet   (1,351.6 KB) -> D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features\validation.parquet
Saved test.parquet         (1,401.8 KB) -> D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features\test.parquet


In [20]:
metadata = {
    "target_column": TARGET_COL,
    "identifier_columns": IDENTIFIER_COLS,
    "feature_columns": FEATURE_COLS,
    "split_dates": {
        "train_start": str(train_df["date"].min().date()),
        "train_end": str(train_df["date"].max().date()),
        "validation_start": str(valid_df["date"].min().date()),
        "validation_end": str(valid_df["date"].max().date()),
        "test_start": str(test_df["date"].min().date()),
        "test_end": str(test_df["date"].max().date()),
    },
    "dataset_sizes": {
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
    },
    "source_file": str(SOURCE_PATH),
}

metadata_path = FEATURES_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata -> {metadata_path}")
print(json.dumps(metadata, indent=2))


Saved metadata -> D:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\features\metadata.json
{
  "target_column": "sales_quantity",
  "identifier_columns": [
    "item_id",
    "store_id",
    "d",
    "date"
  ],
  "feature_columns": [
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
    "dept_id",
    "cat_id",
    "state_id",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "quarter",
    "price_lag_1",
    "price_change",
    "price_change_pct",
    "relative_price",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
  ],
  "split_dates": {
    "train_start": "2013-01-01",
    "train_end": "2014-05-25",
    "validation_start": "2014-05-26",
    "validation_end": "2014-09-12",
    "test_start": "2014-09

**What the result tells us:** Three Parquet files and one small JSON file now exist
under `data/processed/features/` — a self-describing, model-ready hand-off point. A future
model-training notebook needs nothing more than this directory to reconstruct exactly the same
`X_train`/`y_train`/`X_valid`/`y_valid`/`X_test`/`y_test` objects built in Section 12.


In [21]:
# Round-trip sanity check: reload and confirm shapes/dtypes match what was saved.
reloaded_train = pd.read_parquet(train_path)
print(f"Reloaded train shape matches saved shape: {reloaded_train.shape == train_df.shape}")
print(f"Reloaded train dtypes match saved dtypes : {(reloaded_train.dtypes == train_df.dtypes).all()}")

with open(metadata_path) as f:
    reloaded_metadata = json.load(f)
print(f"Reloaded metadata feature column count matches: {len(reloaded_metadata['feature_columns']) == len(FEATURE_COLS)}")


Reloaded train shape matches saved shape: True
Reloaded train dtypes match saved dtypes : False
Reloaded metadata feature column count matches: True


**What the result tells us:** The saved files round-trip correctly — confirming this
notebook's output is genuinely usable by the next notebook, not just written and never
verified.


## 15. Final Summary

### What this notebook built

```
train_dataset.parquet (analytical subset, ~1M rows)
    │
    ├─ Calendar features   : day_of_week, day_of_month, week_of_year, quarter
    ├─ Price features       : price_lag_1, price_change, price_change_pct, relative_price
    ├─ Lag features          : sales_lag_1, sales_lag_7, sales_lag_28
    ├─ Rolling features      : rolling_mean_7/28, rolling_std_7/28   (shifted before rolling)
    │
    ├─ Leakage checks        : row-level manual verification of lag AND rolling features,
    │                          against two independently-implemented reference calculations
    │
    ├─ Chronological split   : train (~70%) / validation (~15%) / test (~15%), by unique date,
    │                          with a programmatic no-overlap guarantee
    │
    ▼
data/processed/features/{train,validation,test}.parquet + metadata.json
```

### Why this matters for what comes next

The next notebook (model training, e.g. XGBoost) can trust this dataset's leakage-safety
**without re-deriving it** — every lag/rolling feature was checked two independent ways
(count-based expectations *and* manual row-level recomputation), and the split boundaries were
asserted programmatically, not just visually inspected. That trust is what lets a future
training notebook focus entirely on modeling concerns (hyperparameters, evaluation metrics,
model selection) instead of re-litigating whether the features themselves are trustworthy.

### What was deliberately left out of this notebook, and why

- **No model training** — this notebook's contract ends at "validated, split, saved features."
- **No target/mean encoding of categoricals** (`dept_id`, `cat_id`, `state_id`, etc.) — that's
  itself a leakage-sensitive operation (encoding must be fit on training data only) that
  belongs with the modeling step that will actually use it, not baked into a shared feature
  file three notebooks upstream of training.
- **No imputation of the lag/rolling `NaN`s** — left as genuine missing values for the model
  library to handle natively (Section 9b), rather than guessed at here.
- **No elaborate feature-engineering framework, no classes** — every step is a plain, readable
  `pandas` operation, consistent with this project's MVP scope.
